# BE Model Behaviour

Sweeps of model parameters and burn-in to understand how each affects behavioural readouts.

**Sections:**
1. 1D parameter sweeps (psychometrics, summary stats, update matrix)
2. 2D parameter sweeps (configurable pair)
3. Burn-in effect (single param, then burn-in × η interaction)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

from Models.BE_core import BEParams, BEState, BEModel
from Data.structures import sample_stimuli, stimulus_to_category
from Analysis.summary_stats import compute_summary_stats, list_available_stats
from Analysis.update_matrix import compute_update_matrix_from_model_trace

print("Available stats:", list_available_stats())


## Config

Change values here. All sweep cells below read from this cell.

In [ ]:
# ── Colour settings ────────────----─────────────────────────────────────────
import matplotlib.colors as mcolors
um_cmap = mcolors.LinearSegmentedColormap.from_list(
	'custom_diverging',
	[(253/255, 120/255, 6/255), (1, 1, 1), (120/255, 0/255, 220/255)]
)

# ── Simulation settings ─────────────────────────────────────────────────────
N_TRIALS  = 500
BURN_IN   = 200      # burn-in applied before each simulated session
SEED      = 42

# ── Baseline parameter values (used as fixed values when sweeping others) ───
BASE_PARAMS = dict(
    sigma_percep = 0.15,
    A_repulsion  = 0.10,
    eta_learning = 0.30,
    eta_relax    = 0.05,
)

# ── Sweep grids (1-D) ────────────────────────────────────────────────────────
SWEEP_VALUES = dict(
    sigma_percep = np.array([0.05, 0.10, 0.15, 0.25, 0.40]),
    A_repulsion  = np.array([0.00, 0.05, 0.10, 0.20, 0.35]),
    eta_learning = np.array([0.01, 0.05, 0.15, 0.30, 0.50, 0.75]),
    eta_relax    = np.array([0.001, 0.01, 0.05, 0.10, 0.20]),
)

# ── Stats to show in sweep plots ────────────────────────────────────────────
SWEEP_STATS = ['accuracy', 'recency', 'win_stay', 'stimulus_sensitivity',
               'choice_entropy', 'perseveration']

# ── 2D sweep config (change PARAM_X / PARAM_Y to switch pair) ────────────────
PARAM_X = 'eta_learning'
PARAM_Y = 'sigma_percep'
GRID_N  = 8        # points per axis  (8×8 = 64 sims)
GRID_STATS_2D = ['accuracy', 'recency', 'win_stay', 'choice_entropy']

GRID_X  = np.linspace(*BEParams.get_bounds()[PARAM_X], GRID_N)
GRID_Y  = np.linspace(*BEParams.get_bounds()[PARAM_Y], GRID_N)

# ── Burn-in sweep config ─────────────────────────────────────────────────────
BURNIN_VALUES     = [0, 50, 100, 200, 500, 1000]
BURNIN_ETA_GRID   = np.array([0.05, 0.15, 0.30, 0.50])   # for interaction plot


## Simulation Helper

In [ ]:
def simulate_session(params_dict, n_trials=N_TRIALS, burn_in=BURN_IN, seed=SEED):
    """
    Simulate one session.
    Returns dict with choices, stimuli, categories, trace.
    """
    params = BEParams(**params_dict)
    rng    = np.random.default_rng(seed)

    # Burn-in
    state = BEState.initial_uniform()
    if burn_in > 0:
        bi_stim = rng.uniform(-1, 1, burn_in)
        bi_cats = stimulus_to_category(bi_stim)
        _, _, state, _ = BEModel.simulate_session(params, state, bi_stim, bi_cats, rng)

    stim = sample_stimuli(n_trials, 'Uniform', rng)
    cats = stimulus_to_category(stim)
    choices, p_B, _, trace = BEModel.simulate_session(
        params, state, stim, cats, rng, return_history=True)

    valid = ~np.isnan(choices)
    stats = compute_summary_stats(
        choices[valid], stim[valid], cats[valid],
        stat_names=SWEEP_STATS, return_dict=True)

    return dict(params=params_dict, choices=choices, stimuli=stim,
                categories=cats, trace=trace, stats=stats, p_B=p_B)


---
## Section 1 · 1D Parameter Sweeps

For each parameter: psychometric curves, key summary stats, and update matrix.

In [ ]:
# ── Run 1D sweeps ─────────────────────────────────────────────────────────────
from Helpers.psychometry import fit_psychometric
from Helpers.utils import cumulative_gaussian

sweep_results = {}
for pname, vals in SWEEP_VALUES.items():
    sweep_results[pname] = []
    for v in vals:
        p = dict(BASE_PARAMS)
        p[pname] = float(v)
        sweep_results[pname].append(simulate_session(p))
    print(f"  {pname}: {len(vals)} points done")
print("All 1D sweeps complete.")


In [ ]:
# ── Plot: psychometric curves per param ──────────────────────────────────────
x_fine = np.linspace(-1, 1, 200)
cmap   = plt.get_cmap('viridis')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Psychometric Curves — 1D Sweeps', fontsize=13, fontweight='bold')

for ax, pname in zip(axes.flat, SWEEP_VALUES):
    vals    = SWEEP_VALUES[pname]
    results = sweep_results[pname]
    colors  = [cmap(i / max(len(vals)-1, 1)) for i in range(len(vals))]

    for res, v, c in zip(results, vals, colors):
        valid = ~np.isnan(res['choices'])
        psych = fit_psychometric(res['stimuli'][valid], res['choices'][valid])
        if psych['success']:
            y = cumulative_gaussian(x_fine, psych['mu'], psych['sigma'],
                                    psych['lapse_low'], psych['lapse_high'])
            ax.plot(x_fine, y, color=c, lw=2, label=f'{v:.3g}')

    ax.axhline(0.5, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.axvline(0,   color='k', lw=0.5, ls='--', alpha=0.4)
    ax.set(xlabel='Stimulus', ylabel='P(B)', title=f'Sweep: {pname}',
           ylim=(-0.05, 1.05))
    ax.legend(title=pname, fontsize=7, title_fontsize=7)

plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: summary stats vs param value ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Summary Statistics — 1D Sweeps', fontsize=13, fontweight='bold')

for ax, pname in zip(axes.flat, SWEEP_VALUES):
    vals    = SWEEP_VALUES[pname]
    results = sweep_results[pname]

    for sname in SWEEP_STATS:
        sv = []
        for res in results:
            val = res['stats'].get(sname, np.nan)
            sv.append(float(val) if not isinstance(val, dict) else np.nan)
        ax.plot(vals, sv, 'o-', lw=1.5, ms=5, label=sname)

    ax.set(xlabel=pname, ylabel='Stat value', title=f'Stats vs {pname}')
    ax.axhline(0, color='k', lw=0.4, ls='--', alpha=0.4)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: update matrices per param ──────────────────────────────────────────
# Show one param at a time; pick the one with the most visible effect first.
PARAM_TO_SHOW = 'eta_learning'   # change to any param name

vals    = SWEEP_VALUES[PARAM_TO_SHOW]
results = sweep_results[PARAM_TO_SHOW]
n_vals  = len(vals)

fig, axes = plt.subplots(1, n_vals, figsize=(3*n_vals, 3.5), sharey=True)
fig.suptitle(f'Update Matrix — sweep {PARAM_TO_SHOW}  (others at baseline)',
             fontsize=12, fontweight='bold')

for ax, res, v in zip(axes, results, vals):
    um = compute_update_matrix_from_model_trace(res['trace'])
    im = ax.imshow(um[0], origin='lower', aspect='auto',
                   cmap=um_cmap, vmin=-0.3, vmax=0.3)
    ax.set_title(f'{PARAM_TO_SHOW}={v:.3g}', fontsize=9)
    ax.set_xlabel('Prev stim bin')
    if ax is axes[0]:
        ax.set_ylabel('Curr stim bin')

fig.colorbar(im, ax=axes[-1], label='ΔP(B)', shrink=0.8)
plt.tight_layout()
plt.show()


In [ ]:
# Re-run this cell with a different PARAM_TO_SHOW to inspect other params
PARAM_TO_SHOW = 'A_repulsion'

vals    = SWEEP_VALUES[PARAM_TO_SHOW]
results = sweep_results[PARAM_TO_SHOW]
n_vals  = len(vals)

fig, axes = plt.subplots(1, n_vals, figsize=(3*n_vals, 3.5), sharey=True)
fig.suptitle(f'Update Matrix — sweep {PARAM_TO_SHOW}  (others at baseline)',
             fontsize=12, fontweight='bold')

for ax, res, v in zip(axes, results, vals):
    um = compute_update_matrix_from_model_trace(res['trace'])
    im = ax.imshow(um[0], origin='lower', aspect='auto',
                   cmap=um_cmap, vmin=-0.3, vmax=0.3)
    ax.set_title(f'{PARAM_TO_SHOW}={v:.3g}', fontsize=9)
    ax.set_xlabel('Prev stim bin')
    if ax is axes[0]:
        ax.set_ylabel('Curr stim bin')

fig.colorbar(im, ax=axes[-1], label='ΔP(B)', shrink=0.8)
plt.tight_layout()
plt.show()


---
## Section 2 · 2D Parameter Sweeps

Change `PARAM_X`, `PARAM_Y` in the Config cell and re-run this section.

All 6 pairs: `(sigma_percep, A_repulsion)`, `(sigma_percep, eta_learning)`, `(sigma_percep, eta_relax)`, `(A_repulsion, eta_learning)`, `(A_repulsion, eta_relax)`, `(eta_learning, eta_relax)`.

In [ ]:
# ── Run 2D grid simulation ────────────────────────────────────────────────────
print(f"2D sweep: {PARAM_X} × {PARAM_Y}  ({GRID_N}×{GRID_N} = {GRID_N**2} sims)")

# Pre-allocate stat arrays: grid_stats_2d[stat_name] -> (GRID_N, GRID_N)
grid_stats_2d = {s: np.full((GRID_N, GRID_N), np.nan) for s in GRID_STATS_2D}

# Store full results (including traces) for later update matrix display
grid_results_2d = [[None]*GRID_N for _ in range(GRID_N)]

for i, vx in enumerate(GRID_X):
    for j, vy in enumerate(GRID_Y):
        p = dict(BASE_PARAMS)
        p[PARAM_X] = float(vx)
        p[PARAM_Y] = float(vy)
        res = simulate_session(p)
        grid_results_2d[j][i] = res   # [row=Y, col=X]
        for sname in GRID_STATS_2D:
            val = res['stats'].get(sname, np.nan)
            if not isinstance(val, dict):
                grid_stats_2d[sname][j, i] = float(val)

print("2D sweep complete. Results stored in grid_results_2d.")


In [ ]:
# ── Plot: heatmaps ────────────────────────────────────────────────────────────
n_stats_2d = len(GRID_STATS_2D)
n_cols_2d  = min(n_stats_2d, 5)
n_rows_2d  = int(np.ceil(n_stats_2d / n_cols_2d))

fig, axes = plt.subplots(n_rows_2d, n_cols_2d,
                          figsize=(4.5*n_cols_2d, 4*n_rows_2d),
                          squeeze=False)
axes_flat_2d = axes.flat
fig.suptitle(f'2D sweep: {PARAM_X} (x) × {PARAM_Y} (y)', fontsize=13, fontweight='bold')

for ax, sname in zip(axes_flat_2d, GRID_STATS_2D):
    data = grid_stats_2d[sname]
    im = ax.imshow(data, origin='lower', aspect='auto',
                   extent=[GRID_X[0], GRID_X[-1], GRID_Y[0], GRID_Y[-1]],
                   cmap=um_cmap)
    ax.set(xlabel=PARAM_X, ylabel=PARAM_Y, title=sname)
    fig.colorbar(im, ax=ax, shrink=0.8)

for ax in list(axes_flat_2d)[n_stats_2d:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: full grid of update matrices ───────────────────────────────────────
# Uses stored grid_results_2d — no re-simulation needed.
fig, axes = plt.subplots(GRID_N, GRID_N,
                          figsize=(2.2*GRID_N, 2.2*GRID_N))
fig.suptitle(f'Update Matrices: {PARAM_X} × {PARAM_Y}',
             fontsize=14, fontweight='bold')

vabs = 0.3   # colour scale for ΔP(B); adjust if needed

for j in range(GRID_N):       # j = Y axis (rows, bottom=low)
    for i in range(GRID_N):   # i = X axis (cols, left=low)
        ax  = axes[GRID_N - 1 - j, i]   # flip row so low-Y is at bottom
        res = grid_results_2d[j][i]

        um, _, _ = compute_update_matrix_from_model_trace(res['trace'])
        im = ax.imshow(um, origin='lower', aspect='auto',
                       cmap=um_cmap, vmin=-vabs, vmax=vabs)
        ax.set_xticks([]); ax.set_yticks([])

        # Label edges only
        if j == 0:
            ax.set_xlabel(f'{GRID_X[i]:.2g}', fontsize=7)
        if i == 0:
            ax.set_ylabel(f'{GRID_Y[j]:.2g}', fontsize=7, rotation=0, labelpad=20)

# Shared colourbar
# fig.colorbar(im, ax=axes, label='ΔP(B)', shrink=0.6, pad=0.02)

# Axis labels

fig.supxlabel(f'{PARAM_X}', fontsize=11, fontweight='bold')
fig.supylabel(f'{PARAM_Y}', fontsize=11, fontweight='bold')

plt.tight_layout(rect=[0.01, 0.01, 1, 0.99])
plt.show()


---
## Section 3 · Burn-in Effect

### 3a. Burn-in sweep at fixed params
How many warm-up trials are needed before single-session statistics stabilise?

In [ ]:
# ── Sweep burn-in at BASE_PARAMS ──────────────────────────────────────────────
burnin_results = []
for bi in BURNIN_VALUES:
    res = simulate_session(BASE_PARAMS, burn_in=bi)
    res['burn_in'] = bi
    burnin_results.append(res)

print("Burn-in sweep done:", BURNIN_VALUES)


In [ ]:
# ── Plot: stats vs burn-in ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(f'Effect of Burn-in  (BASE_PARAMS: η={BASE_PARAMS["eta_learning"]}, '
             f'σ={BASE_PARAMS["sigma_percep"]})', fontsize=12, fontweight='bold')

for ax, sname in zip(axes.flat, SWEEP_STATS):
    vals = [float(r['stats'].get(sname, np.nan)) for r in burnin_results]
    ax.plot(BURNIN_VALUES, vals, 'o-', lw=2, ms=6, color='steelblue')
    ax.axhline(vals[-1], color='red', lw=1, ls='--', alpha=0.5, label='asymptote')
    ax.set(xlabel='Burn-in trials', ylabel=sname, title=sname)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: update matrix vs burn-in ────────────────────────────────────────────
n = len(BURNIN_VALUES)
fig, axes = plt.subplots(1, n, figsize=(3*n, 3.5), sharey=True)
fig.suptitle('Update Matrix vs Burn-in  (BASE_PARAMS)', fontsize=11, fontweight='bold')

for ax, res in zip(axes, burnin_results):
    um = compute_update_matrix_from_model_trace(res['trace'])
    im = ax.imshow(um[0], origin='lower', aspect='auto',
                   cmap=um_cmap, vmin=-0.3, vmax=0.3)
    ax.set_title(f"burn_in={res['burn_in']}", fontsize=8)
    ax.set_xlabel('Prev stim bin')
    if ax is axes[0]:
        ax.set_ylabel('Curr stim bin')

fig.colorbar(im, ax=axes[-1], label='ΔP(B)', shrink=0.8)
plt.tight_layout()
plt.show()


### 3b. Burn-in × η interaction

Does burn-in matter more for high-η (naive) animals or low-η (expert) animals?

In [ ]:
# ── Burn-in × eta grid ────────────────────────────────────────────────────────
# For each (burn_in, eta) pair, simulate and record stats.
eta_vals = BURNIN_ETA_GRID
bi_vals  = BURNIN_VALUES

# interaction_stats[sname] -> (len(bi_vals), len(eta_vals))
interaction_stats = {s: np.full((len(bi_vals), len(eta_vals)), np.nan)
                     for s in SWEEP_STATS}

for j, eta in enumerate(eta_vals):
    for i, bi in enumerate(bi_vals):
        p = dict(BASE_PARAMS)
        p['eta_learning'] = float(eta)
        res = simulate_session(p, burn_in=bi)
        for sname in SWEEP_STATS:
            val = res['stats'].get(sname, np.nan)
            if not isinstance(val, dict):
                interaction_stats[sname][i, j] = float(val)

print("Burn-in × eta interaction complete.")


In [ ]:
# ── Plot: interaction ─────────────────────────────────────────────────────────
STAT_TO_SHOW = 'recency'   # change to any stat in SWEEP_STATS

data   = interaction_stats[STAT_TO_SHOW]
cmap_i = plt.cm.plasma
colors = [cmap_i(k / max(len(eta_vals)-1, 1)) for k in range(len(eta_vals))]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Burn-in × η interaction  [{STAT_TO_SHOW}]',
             fontsize=12, fontweight='bold')

# Left: stat vs burn_in, one line per eta
ax = axes[0]
for j, eta in enumerate(eta_vals):
    ax.plot(bi_vals, data[:, j], 'o-', color=colors[j], lw=2, ms=5,
            label=f'η={eta:.2f}')
ax.set(xlabel='Burn-in trials', ylabel=STAT_TO_SHOW,
       title='Stat vs burn-in, grouped by η')
ax.legend(fontsize=8)

# Right: heatmap
ax = axes[1]
im = ax.imshow(data, origin='lower', aspect='auto',
               extent=[eta_vals[0], eta_vals[-1], bi_vals[0], bi_vals[-1]],
               cmap=um_cmap)
ax.set(xlabel='η_learning', ylabel='Burn-in trials',
       title='Heatmap (colour = stat value)')
fig.colorbar(im, ax=ax, label=STAT_TO_SHOW, shrink=0.85)

plt.tight_layout()
plt.show()

# Re-run with different STAT_TO_SHOW to inspect other stats


### 3c. Burn-in × Model Param — 2D Grid

Change `BURNIN_PARAM` to any of the 4 model parameters. Re-run the two cells below to switch pair.

In [ ]:
# ── Config for burn-in × param grid ──────────────────────────────────────────
BURNIN_PARAM     = 'eta_learning'   # change to any param: sigma_percep, A_repulsion, eta_relax
BURNIN_PARAM_N   = 7                # points along the param axis
BURNIN_GRID_N    = len(BURNIN_VALUES)   # reuse BURNIN_VALUES from Section 3 config

BURNIN_PARAM_GRID = np.linspace(*BEParams.get_bounds()[BURNIN_PARAM], BURNIN_PARAM_N)

# Stats to show as heatmaps (reuses GRID_STATS_2D from Section 2)
BURNIN_GRID_STATS = GRID_STATS_2D

print(f"Burn-in × {BURNIN_PARAM}: {BURNIN_GRID_N} × {BURNIN_PARAM_N} = "
      f"{BURNIN_GRID_N * BURNIN_PARAM_N} sims")

# ── Run grid ──────────────────────────────────────────────────────────────────
# bi_grid_stats[sname] -> (BURNIN_GRID_N, BURNIN_PARAM_N)  rows=burn-in, cols=param
bi_grid_stats   = {s: np.full((BURNIN_GRID_N, BURNIN_PARAM_N), np.nan) for s in BURNIN_GRID_STATS}
bi_grid_results = [[None]*BURNIN_PARAM_N for _ in range(BURNIN_GRID_N)]

for i, bi in enumerate(BURNIN_VALUES):
    for j, pv in enumerate(BURNIN_PARAM_GRID):
        p = dict(BASE_PARAMS)
        p[BURNIN_PARAM] = float(pv)
        res = simulate_session(p, burn_in=int(bi))
        bi_grid_results[i][j] = res
        for sname in BURNIN_GRID_STATS:
            val = res['stats'].get(sname, np.nan)
            if not isinstance(val, dict):
                bi_grid_stats[sname][i, j] = float(val)

print("Done.")


In [ ]:
# ── Plot: stat heatmaps ───────────────────────────────────────────────────────
n_st   = len(BURNIN_GRID_STATS)
n_cols = min(n_st, 5)
n_rows = int(np.ceil(n_st / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5*n_cols, 4*n_rows), squeeze=False)
fig.suptitle(f'Burn-in (y) × {BURNIN_PARAM} (x) — stat heatmaps',
             fontsize=12, fontweight='bold')

for ax, sname in zip(axes.flat, BURNIN_GRID_STATS):
    im = ax.imshow(bi_grid_stats[sname], origin='lower', aspect='auto',
                   extent=[BURNIN_PARAM_GRID[0], BURNIN_PARAM_GRID[-1],
                           BURNIN_VALUES[0],      BURNIN_VALUES[-1]],
                   cmap=um_cmap)
    ax.set(xlabel=BURNIN_PARAM, ylabel='Burn-in trials', title=sname)
    fig.colorbar(im, ax=ax, shrink=0.8)

for ax in list(axes.flat)[n_st:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: update matrix grid (burn-in rows × param cols) ─────────────────────
fig, axes = plt.subplots(BURNIN_GRID_N, BURNIN_PARAM_N,
                          figsize=(2.2*BURNIN_PARAM_N, 2.2*BURNIN_GRID_N))
fig.suptitle(f'Update Matrices: {BURNIN_PARAM} × Burn-in',
             fontsize=12, fontweight='bold')

vabs = 0.3

for i, bi in enumerate(BURNIN_VALUES):         # i = burn-in row
    for j, pv in enumerate(BURNIN_PARAM_GRID): # j = param col
        ax  = axes[BURNIN_GRID_N - 1 - i, j]  # flip so low burn-in at bottom
        res = bi_grid_results[i][j]
        um, _, _ = compute_update_matrix_from_model_trace(res['trace'])
        im = ax.imshow(um, origin='lower', aspect='auto',
                       cmap=um_cmap, vmin=-vabs, vmax=vabs)
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_xlabel(f'{pv:.2g}', fontsize=7)
        if j == 0:
            ax.set_ylabel(f'{int(bi)}', fontsize=7, rotation=0, labelpad=22)

# fig.colorbar(im, ax=axes, label='ΔP(B)', shrink=0.6)
fig.supxlabel(BURNIN_PARAM, fontsize=11, fontweight='bold')
fig.supylabel('Burn-in trials', fontsize=11, fontweight='bold')
plt.tight_layout(rect=[0.01, 0.01, 1, 0.99])
plt.show()


---
## Section 4 · Param–Stat Correlation Analysis

Sample parameters randomly from the uniform prior, simulate one session each, compute all stats.
Then compute Spearman correlations between each param and each stat.

Answers:
- Which stats carry information about which parameter?
- Which stats are redundant with each other?
- Which stats uniquely identify a specific parameter (useful for SBI)?

This is complementary to the sweeps above: sweeps show the *shape* of the relationship; this shows the *strength and uniqueness*.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
N_SAMPLES_CORR = 2000    # number of random simulations (more = smoother)
N_TRIALS_CORR  = 500
BURN_IN_CORR   = 200
SEED_CORR      = 99

# All stats to evaluate (everything registered)
ALL_STATS = list_available_stats()

from Analysis.summary_stats import get_stat_names_expanded
EXPANDED_NAMES = get_stat_names_expanded(ALL_STATS)
N_STATS = len(EXPANDED_NAMES)
PARAM_NAMES = ['sigma_percep', 'A_repulsion', 'eta_learning', 'eta_relax']

print(f"Stat groups: {len(ALL_STATS)}  →  {N_STATS} expanded features")
print(f"Will run {N_SAMPLES_CORR} simulations...")


In [ ]:
import time
from Analysis.summary_stats import compute_summary_stats, flatten_stats

rng_corr   = np.random.default_rng(SEED_CORR)
bounds     = BEParams.get_bounds()

param_values = np.empty((N_SAMPLES_CORR, 4))
stat_values  = np.empty((N_SAMPLES_CORR, N_STATS))
stat_values[:] = np.nan

t0 = time.time()
n_failed = 0

for k in range(N_SAMPLES_CORR):
    if k % 500 == 0 and k > 0:
        elapsed = time.time() - t0
        print(f"  {k}/{N_SAMPLES_CORR}  ({elapsed:.0f}s elapsed, "
              f"~{(N_SAMPLES_CORR - k) / (k / elapsed):.0f}s left)")

    # Sample uniformly from prior
    params = BEParams(
        sigma_percep = rng_corr.uniform(*bounds['sigma_percep']),
        A_repulsion  = rng_corr.uniform(*bounds['A_repulsion']),
        eta_learning = rng_corr.uniform(*bounds['eta_learning']),
        eta_relax    = rng_corr.uniform(*bounds['eta_relax']),
    )
    param_values[k] = params.to_array()

    # Burn-in
    state = BEState.initial_uniform()
    if BURN_IN_CORR > 0:
        bi_stim = rng_corr.uniform(-1, 1, BURN_IN_CORR)
        bi_cats = stimulus_to_category(bi_stim)
        _, _, state, _ = BEModel.simulate_session(params, state, bi_stim, bi_cats, rng_corr)

    stim = sample_stimuli(N_TRIALS_CORR, 'Uniform', rng_corr)
    cats = stimulus_to_category(stim)
    choices, _, _, _ = BEModel.simulate_session(params, state, stim, cats, rng_corr)

    try:
        stats_dict = compute_summary_stats(
            choices, stim, cats,
            stat_names=ALL_STATS, return_dict=True)
        stat_values[k] = flatten_stats(stats_dict)
    except Exception:
        n_failed += 1

print(f"Done. {n_failed} failures out of {N_SAMPLES_CORR}.")
print(f"Total time: {time.time() - t0:.0f}s")

# Drop rows with all-NaN stats
valid_rows = ~np.all(np.isnan(stat_values), axis=1)
param_values_clean = param_values[valid_rows]
stat_values_clean  = stat_values[valid_rows]
print(f"Valid simulations: {valid_rows.sum()}")


In [ ]:
# ── Compute Spearman correlations ─────────────────────────────────────────────
from scipy.stats import spearmanr

param_stat_corr = np.full((4, N_STATS), np.nan)
param_stat_pval = np.full((4, N_STATS), np.nan)

for p in range(4):
    for s in range(N_STATS):
        sv   = stat_values_clean[:, s]
        mask = ~np.isnan(sv)
        if mask.sum() >= 30:
            rho, pval = spearmanr(param_values_clean[mask, p], sv[mask])
            param_stat_corr[p, s] = rho
            param_stat_pval[p, s] = pval

# Inter-stat correlations
stat_stat_corr = np.full((N_STATS, N_STATS), np.nan)
for i in range(N_STATS):
    for j in range(N_STATS):
        vi, vj = stat_values_clean[:, i], stat_values_clean[:, j]
        mask   = ~(np.isnan(vi) | np.isnan(vj))
        if mask.sum() >= 30:
            rho, _ = spearmanr(vi[mask], vj[mask])
            stat_stat_corr[i, j] = rho

print("Correlations computed.")


In [ ]:
# ── Plot: param–stat correlation heatmap ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(14, N_STATS * 0.45), 5))

im = ax.imshow(param_stat_corr, aspect='auto', cmap=um_cmap, vmin=-1, vmax=1)
ax.set_yticks(range(4));     ax.set_yticklabels(PARAM_NAMES, fontsize=9)
ax.set_xticks(range(N_STATS)); ax.set_xticklabels(EXPANDED_NAMES, rotation=90, fontsize=7)

# Annotate cells with |rho| > 0.1
for i in range(4):
    for j in range(N_STATS):
        v = param_stat_corr[i, j]
        if not np.isnan(v) and abs(v) > 0.10:
            txt_col = 'white' if abs(v) > 0.5 else 'black'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=5.5, color=txt_col)

fig.colorbar(im, ax=ax, shrink=0.8, label='Spearman ρ')
ax.set_title('Parameter–Statistic Correlations  (random prior samples, single session)',
             fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: inter-stat redundancy heatmap ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(10, N_STATS * 0.42), max(9, N_STATS * 0.38)))

im = ax.imshow(stat_stat_corr, aspect='auto', cmap=um_cmap, vmin=-1, vmax=1)
ax.set_xticks(range(N_STATS)); ax.set_xticklabels(EXPANDED_NAMES, rotation=90, fontsize=6)
ax.set_yticks(range(N_STATS)); ax.set_yticklabels(EXPANDED_NAMES, fontsize=6)

fig.colorbar(im, ax=ax, shrink=0.8, label='Spearman ρ')
ax.set_title('Inter-Stat Redundancy Matrix', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: per-parameter discriminability ──────────────────────────────────────
# For each param: bars show |corr with this param| (blue) vs
# max|corr with any other param| (red, negated).
# Green star = uniquely informative (strong for this param, weak for others).
TOP_N_SHOW = 25   # how many stats to show per param (sorted by |corr|)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Per-Parameter Stat Discriminability(* = uniquely informative)',
             fontsize=12, fontweight='bold')

for p, (ax, pname) in enumerate(zip(axes.flat, PARAM_NAMES)):
    abs_this   = np.abs(param_stat_corr[p])
    other_mask = np.ones(4, dtype=bool); other_mask[p] = False
    abs_others = np.nanmax(np.abs(param_stat_corr[other_mask]), axis=0)
    uniqueness = abs_this - abs_others

    order  = np.argsort(abs_this)[::-1][:TOP_N_SHOW]
    names  = [EXPANDED_NAMES[i] for i in order]
    y      = np.arange(len(order))

    ax.barh(y,  abs_this[order],   color='#1976d2', alpha=0.75, label='|ρ| with this param')
    ax.barh(y, -abs_others[order], color='#d32f2f', alpha=0.50, label='−max|ρ| with others')

    for yi, idx in enumerate(order):
        if uniqueness[idx] > 0.05 and abs_this[idx] > 0.15:
            ax.text(abs_this[idx] + 0.01, yi, '*', fontsize=13,
                    color='green', va='center', fontweight='bold')

    ax.set_yticks(y); ax.set_yticklabels(names, fontsize=7)
    ax.set_xlabel('|Spearman ρ|'); ax.set_title(pname, fontsize=10)
    ax.axvline(0, color='k', lw=0.5)
    ax.invert_yaxis()
    if p == 0:
        ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.show()


In [ ]:
# ── Summary table: stat usefulness ranking ────────────────────────────────────
# Sorted by |correlation with eta_learning| (the primary parameter of interest)
rows = []
eta_idx = 2   # eta_learning is index 2 in PARAM_NAMES

for s in range(N_STATS):
    abs_corrs  = np.abs(param_stat_corr[:, s])
    best_p     = int(np.nanargmax(abs_corrs))
    redundancy = np.nanmean(np.abs(stat_stat_corr[s]))

    rows.append(dict(
        stat       = EXPANDED_NAMES[s],
        best_param = PARAM_NAMES[best_p],
        best_rho   = param_stat_corr[best_p, s],
        eta_rho    = param_stat_corr[eta_idx, s],
        redundancy = redundancy,
    ))

rows.sort(key=lambda r: abs(r['eta_rho']) if not np.isnan(r['eta_rho']) else 0, reverse=True)

print(f"{'Stat':<24s} {'Best param':<14s} {'best_rho':>8s} {'eta_rho':>8s} {'redundancy':>10s}  note")
print('-' * 80)
for r in rows:
    note = ''
    if abs(r['eta_rho']) > 0.3:
        note = '<-- good for eta'
    elif abs(r['best_rho']) > 0.3 and abs(r['eta_rho']) < 0.15:
        note = f'<-- good for {r["best_param"]}'
    print(f"{r['stat']:<24s} {r['best_param']:<14s} {r['best_rho']:>+8.3f} "
          f"{r['eta_rho']:>+8.3f} {r['redundancy']:>10.3f}  {note}")

print()
print("Columns:")
print("  best_rho   = ρ with the single most correlated parameter")
print("  eta_rho    = ρ with eta_learning specifically")
print("  redundancy = mean |ρ| with all other stats (higher = more redundant)")
